# 📌 Objective
This project aims to automatically determine which of two AI-generated responses is better for a given prompt using supervised text classification. The model is trained using labeled comparisons and leverages a custom text cleaning pipeline and XGBoost classifier.

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from nltk.corpus import stopwords

In [ ]:
# Manual stopwords list (minimal English stopwords)
stop_words = set("""
a about above after again against all am an and any are aren't as at be because been before being below between both but by
could couldn't did didn't do does doesn't doing don't down during each few for from further had hadn't has hasn't have haven't
having he he'd he'll he's her here here's hers herself him himself his how how's i i'd i'll i'm i've if in into is isn't it
it's its itself let's me more most mustn't my myself no nor not of off on once only or other ought our ours ourselves out over
own same shan't she she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then
there there's these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll
we're we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves
""".split())

# === 1. Load Data ===
train = pd.read_csv("/kaggle/input/model-data/train.csv")
test = pd.read_csv("/kaggle/input/model-data/test.csv")

# === 2. Create Labels ===
train['label'] = train[['winner_model_a', 'winner_model_b', 'winner_tie']].values.argmax(axis=1)

# === 3. Combine Text Columns ===
train['full_text'] = train['prompt'] + " " + train['response_a'] + " " + train['response_b']
test['full_text'] = test['prompt'] + " " + test['response_a'] + " " + test['response_b']

# === 4. Text Cleaning Class ===
class TextCleaner(BaseEstimator, TransformerMixin):
    def clean_text(self, text):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation
        tokens = text.split()
        tokens = [t for t in tokens if t not in stop_words]  # Remove stopwords
        return ' '.join(tokens)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.apply(self.clean_text)

# === 5. Define Pipeline ===
pipeline = Pipeline([
    ('cleaner', TextCleaner()),
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('clf', XGBClassifier(
        use_label_encoder=False,
        eval_metric='mlogloss',
        max_depth=6,
        n_estimators=300,
        learning_rate=0.1,
        subsample=0.8
    ))
])


In [ ]:
# === 6. Train the Model ===
pipeline.fit(train['full_text'], train['label'])

# === 7. Training Predictions ===
train_preds = pipeline.predict(train['full_text'])


In [ ]:
# === 8. Confusion Matrix ===
labels = ["Model A Wins", "Model B Wins", "Tie"]
plt.figure(figsize=(6, 5))
conf_mat = confusion_matrix(train['label'], train_preds)
sns.heatmap(conf_mat, annot=True, fmt='d', cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix: Which Model Was Predicted as Better?")
plt.tight_layout()
plt.show()


In [ ]:
# === 9. Prediction Distribution ===
unique, counts = np.unique(train_preds, return_counts=True)
plt.figure(figsize=(6, 4))
sns.barplot(x=[labels[i] for i in unique], y=counts)
plt.title("Which Answer Did the Model Choose Most Often?")
plt.ylabel("Number of Predictions")
plt.xticks(rotation=15)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
# === 10. Precision, Recall, F1 ===
report = classification_report(train['label'], train_preds, output_dict=True)
scores = pd.DataFrame(report).transpose().iloc[:3][['precision', 'recall', 'f1-score']]
scores.index = labels
scores.plot(kind="bar", figsize=(8, 5))
plt.title("Model Performance by Class")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
# === 1. Combine text fields ===
test['full_text'] = test['prompt'] + " " + test['response_a'] + " " + test['response_b']

# === 2. Predict class probabilities ===
proba_preds = pipeline.predict_proba(test['full_text'])

# === 3. Create submission DataFrame in required format
submission = pd.DataFrame(proba_preds, columns=["winner_model_a", "winner_model_b", "winner_tie"])
submission["id"] = test["id"]

# === 4. Rearrange columns to match expected format
submission = submission[["id", "winner_model_a", "winner_model_b", "winner_tie"]]

# === 5. Save submission file
submission.to_csv("submission.csv", index=False)
print("✅ Submission file with probabilities has been created successfully.")



In [ ]:
# Save the trained pipeline
joblib.dump(pipeline, "trained_pipeline_model.pkl")
print("✅ The trained pipeline model has been saved successfully as 'trained_pipeline_model.pkl'.")

# 📊 1. Confusion Matrix: Which Model Was Predicted as Better?

This matrix shows the distribution of predictions versus actual outcomes.

Diagonal cells represent correct predictions (e.g., 15,356 for "Model A Wins").

Off-diagonal cells represent misclassifications.

Most common mistakes: confusing ties with wins.

![](/kaggle/input/grafikler/1.png)

# 📈 2. Which Answer Did the Model Choose Most Often?

This bar chart displays the total number of predictions made for each class.

Model A Wins was the most predicted class.

This may suggest a slight model bias or reflect dataset distribution.

![](/kaggle/input/grafikler/2.png)

# 📐 3. Model Performance by Class (Precision / Recall / F1-Score)

The performance metrics are broken down per class:

Class	Precision	Recall	F1-Score
Model A Wins	~0.71	~0.76	~0.74
Model B Wins	~0.76	~0.71	~0.74
Tie	~0.67	~0.66	~0.66

Precision: Accuracy of positive predictions.

Recall: Ability to find all relevant instances.

F1-Score: Harmonic mean of precision and recall.

Insight: The model performs slightly better on "Model A Wins" and "Model B Wins" compared to "Tie" cases, which is expected given the more ambiguous nature of tie outcomes.

![](/kaggle/input/grafikler/3.png)

# ✅ Summary
The model exhibits strong performance across all three classes.

Slight confusion between ties and wins is observed, indicating an area for future improvement.

Overall, the classifier achieves a balanced trade-off between precision and recall.